# Qwen3-TTS | Servidor Backend en la Nube (Google Colab)

Este script levantará el motor de Inteligencia Artificial **Qwen3-TTS** usando la GPU gratuita de Google Colab y te dará un **Enlace Público**.

Ese enlace lo copiarás y pegarás en tu Frontend local para conectar ambos sistemas.

### ⚠️ PASO MUY IMPORTANTE ANTES DE INICIAR:
Ve al menú superior: **Entorno de ejecución > Cambiar tipo de entorno de ejecución** y asegúrate de que el **Acelerador de hardware** esté en **T4 GPU**.

## 1. Descargar el Servidor e Instalar Dependencias

In [ ]:
# 1. Configuramos el entorno y descargamos el código necesario
import os

# Creamos la estructura de carpetas (similar a la local)
!mkdir -p /content/Qwen3-TTS-Platform/backend/app

# Descargamos Qwen-TTS y requerimientos del framework
!pip install fastapi uvicorn[standard] python-multipart qwen-tts soundfile numpy pydantic

# Instalar pyngrok para exponer el servidor a internet
!pip install pyngrok nest-asyncio

## 2. Inyectar el Código del Backend

In [ ]:
%%writefile /content/Qwen3-TTS-Platform/backend/app/config.py
from dataclasses import dataclass, field
from typing import Dict, List

MODEL_REGISTRY: Dict[str, Dict[str, str]] = {
    "fast": {
        "custom_voice": "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice",
        "voice_clone":  "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    },
    "quality": {
        "custom_voice":  "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
        "voice_clone":   "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        "voice_design":  "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    },
}

SUPPORTED_QUALITIES = ["fast", "quality"]
SUPPORTED_MODES = ["custom_voice", "voice_clone", "voice_design"]
SUPPORTED_LANGUAGES = ["Auto", "Chinese", "English", "Japanese", "Korean", "German", "French", "Russian", "Portuguese", "Spanish", "Italian"]
DEFAULT_SPEAKERS = ["Vivian", "Ryan", "Aria", "Emily", "Owen", "Rina", "Hudson", "Claire", "Haruto", "Stella"]

@dataclass
class ServerConfig:
    host: str = "0.0.0.0"
    port: int = 8000
    cors_origins: List[str] = field(default_factory=lambda: ["*"])
    default_quality: str = "fast"

def get_model_id(quality: str, mode: str) -> str:
    return MODEL_REGISTRY[quality.lower()][mode.lower()]

In [ ]:
%%writefile /content/Qwen3-TTS-Platform/backend/app/models.py
from typing import Optional, List
from pydantic import BaseModel, Field
from .config import SUPPORTED_QUALITIES, SUPPORTED_MODES, SUPPORTED_LANGUAGES, DEFAULT_SPEAKERS

class GenerateAudioRequest(BaseModel):
    text: str = Field(..., max_length=5000)
    language: str = Field(default="Auto")
    mode: str = Field(default="custom_voice")
    quality: str = Field(default="fast")
    speaker: Optional[str] = Field(default="Vivian")
    instruct: Optional[str] = Field(default=None)
    ref_audio_base64: Optional[str] = Field(default=None)
    ref_text: Optional[str] = Field(default=None)

class GenerateAudioResponse(BaseModel):
    success: bool
    audio_base64: Optional[str] = None
    sample_rate: Optional[int] = None
    duration_seconds: Optional[float] = None
    model_used: Optional[str] = None
    message: Optional[str] = None

class HealthResponse(BaseModel):
    status: str = "ok"
    gpu_available: bool
    gpu_name: Optional[str] = None
    vram_total_gb: Optional[float] = None
    vram_free_gb: Optional[float] = None

In [ ]:
%%writefile /content/Qwen3-TTS-Platform/backend/app/tts_engine.py
import gc, io, base64, logging, tempfile, threading, os
from typing import Optional, Tuple
import numpy as np
import soundfile as sf

logger = logging.getLogger(__name__)
_torch = None
_Qwen3TTSModel = None

def _ensure_imports():
    global _torch, _Qwen3TTSModel
    if _torch is None: import torch; _torch = torch
    if _Qwen3TTSModel is None: from qwen_tts import Qwen3TTSModel; _Qwen3TTSModel = Qwen3TTSModel

class TTSEngine:
    def __init__(self):
        self._lock = threading.Lock()
        self._model = None

    def _load_model(self, model_id: str):
        _ensure_imports()
        device = "cuda:0" if _torch.cuda.is_available() else "cpu"
        self._model = _Qwen3TTSModel.from_pretrained(model_id, device_map=device, dtype=_torch.bfloat16, attn_implementation="eager")

    def _unload_model(self):
        _ensure_imports()
        if self._model is not None:
            del self._model
            self._model = None
        gc.collect()
        if _torch.cuda.is_available(): _torch.cuda.empty_cache()

    def generate(self, model_id: str, mode: str, text: str, language: str = "Auto", speaker: Optional[str] = None, instruct: Optional[str] = None, ref_audio_base64: Optional[str] = None, ref_text: Optional[str] = None):
        _ensure_imports()
        with self._lock:
            try:
                self._load_model(model_id)
                if mode == "custom_voice":
                    k = {"text":text, "language":language, "speaker":speaker}
                    if instruct: k["instruct"] = instruct
                    wavs, sr = self._model.generate_custom_voice(**k)
                elif mode == "voice_clone":
                    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                        tmp.write(base64.b64decode(ref_audio_base64)); tmp_path = tmp.name
                    k = {"text":text, "language":language, "ref_audio":tmp_path}
                    if ref_text: k["ref_text"] = ref_text
                    else: k["x_vector_only_mode"] = True
                    wavs, sr = self._model.generate_voice_clone(**k)
                    os.unlink(tmp_path)
                elif mode == "voice_design":
                    wavs, sr = self._model.generate_voice_design(text=text, language=language, instruct=instruct)
                
                buffer = io.BytesIO(); sf.write(buffer, wavs[0], sr, format="WAV"); buffer.seek(0)
                return base64.b64encode(buffer.read()).decode("utf-8"), sr, len(wavs[0])/sr
            finally:
                self._unload_model()
                
    @staticmethod
    def get_gpu_info() -> dict:
        _ensure_imports()
        if not _torch.cuda.is_available(): return {"available": False}
        try:
            return {
                "available": True, "name": _torch.cuda.get_device_name(0),
                "vram_total_gb": _torch.cuda.get_device_properties(0).total_mem / (1024**3),
                "vram_free_gb": (_torch.cuda.get_device_properties(0).total_mem - _torch.cuda.memory_allocated(0)) / (1024**3),
            }
        except: return {"available": True, "name": "Unknown"}
engine = TTSEngine()

In [ ]:
%%writefile /content/Qwen3-TTS-Platform/backend/app/main.py
import logging
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from .config import MODEL_REGISTRY, ServerConfig, get_model_id, SUPPORTED_LANGUAGES, DEFAULT_SPEAKERS
from .models import GenerateAudioRequest, GenerateAudioResponse, HealthResponse
from .tts_engine import engine

config = ServerConfig()
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.get("/health", response_model=HealthResponse)
async def health_check():
    gpu = engine.get_gpu_info()
    return HealthResponse(gpu_available=gpu["available"], gpu_name=gpu.get("name"), vram_total_gb=gpu.get("vram_total_gb"), vram_free_gb=gpu.get("vram_free_gb"))

@app.get("/models")
async def list_models(): return {"models": []}

@app.get("/speakers")
async def list_speakers(): return {"speakers": DEFAULT_SPEAKERS}

@app.get("/languages")
async def list_languages(): return {"languages": SUPPORTED_LANGUAGES}

@app.post("/generate_audio", response_model=GenerateAudioResponse)
async def generate_audio(request: GenerateAudioRequest):
    try:
        model_id = get_model_id(request.quality, request.mode)
        audio_b64, sr, dur = engine.generate(model_id, request.mode, request.text, request.language, request.speaker, request.instruct, request.ref_audio_base64, request.ref_text)
        return GenerateAudioResponse(success=True, audio_base64=audio_b64, sample_rate=sr, duration_seconds=dur, model_used=model_id)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

## 3. Iniciar Túnel Ngrok y Servidor Backend
Para conectar el frontend de tu computadora con este servidor de Colab, usaremos **ngrok**.
Necesitas tener una cuenta gratis en ngrok.com y pegar tu **Authtoken** aquí abajo (reemplaza `TU_TOKEN_AQUI`).

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import os
import sys
sys.path.append('/content/Qwen3-TTS-Platform')

# ⚠️ REEMPLAZA ESTO CON TU NGROK AUTHTOKEN ⚠️
# Consiguelo gratis en: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "TU_TOKEN_AQUI"

if NGROK_AUTH_TOKEN == "TU_TOKEN_AQUI":
    print("❌ ERROR: Debes poner tu Token de Ngrok en el código primero.")
else:
    # Matar procesos existentes en el puerto 8000 si reinicias la celda
    os.system('fuser -k 8000/tcp')
    
    # Autenticar ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    
    # Importante: Desconectar túneles activos de sesiones anteriores
    try:
        for tunnel in ngrok.get_tunnels():
            ngrok.disconnect(tunnel.public_url)
    except:
        pass
        
    # Crear nuevo túnel público al puerto 8000
    public_url = ngrok.connect(8000).public_url
    
    print("\n" + "="*60)
    print("✅ SERVIDOR BACKEND ACTIVO EN LA NUBE ✅")
    print("\nCopia este enlace exacto y pégalo en 'URL del Backend' en la página de Configuración de tu Frontend:\n")
    print(f"   {public_url}   ")
    print("\n" + "="*60 + "\n")
    
    # Permitir a asyncio correr servidores anidados (requerido por colab)
    nest_asyncio.apply()
    
    print("FastAPI logs:\n")
    # Iniciar el backend
    uvicorn.run("backend.app.main:app", host="0.0.0.0", port=8000)